In [1]:
# packages
import pandas as pd
from mod02_build_bot_predictor import train_model

### Define a function to extract predictions from the model

In [2]:
def predict_bot(df, model=None):
    """
    Predict whether each account is a bot (1) or human (0).
    """
    if model is None:
        model = train_model()

    preds = model.predict(df)
    return pd.Series(preds, index=df.index)

### Define a function to evaluate model error

In [3]:
def confusion_matrix_and_metrics(y_true, y_pred):
    """
    Computes confusion matrix and common error rates for binary classification.

    Assumes labels:
      0 = negative class
      1 = positive class

    Returns:
      dict with:
        tn, fp, fn, tp
        misclassification_rate
        false_positive_rate
        false_negative_rate
    """
    tn = fp = fn = tp = 0

    for yt, yp in zip(y_true, y_pred):
        if yt == 0 and yp == 0:
            tn += 1
        elif yt == 0 and yp == 1:
            fp += 1
        elif yt == 1 and yp == 0:
            fn += 1
        elif yt == 1 and yp == 1:
            tp += 1
        else:
            raise ValueError("Labels must be 0 or 1")

    total = tn + fp + fn + tp

    misclassification_rate = (fp + fn) / total if total > 0 else 0.0
    false_positive_rate = fp / (fp + tn) if (fp + tn) > 0 else 0.0
    false_negative_rate = fn / (fn + tp) if (fn + tp) > 0 else 0.0

    return {
        "tp": tp,
        "tn": tn,
        "fp": fp,
        "fn": fn,
        "misclassification_rate": misclassification_rate,
        "false_positive_rate": false_positive_rate,
        "false_negative_rate": false_negative_rate,
    }


### Load the data

In [4]:
TRAIN_PATH = "mod02_data/train.csv"
train = pd.read_csv(TRAIN_PATH)

TEST_PATH = "mod02_data/test.csv"
test = pd.read_csv(TEST_PATH)

### Format the data by independent vs. dependent variables

In [5]:
X_train = train.drop(columns=["is_bot"])
y_train = train['is_bot']

X_test = test.drop(columns=["is_bot"])
y_test = test['is_bot']

### Build the model on training data

In [6]:
model = train_model(X_train, y_train)

### Get the model predictions on training and test data

In [7]:
y_pred_train = predict_bot(X_train, model)
y_pred_test = predict_bot(X_test, model)

### Check results on the training set (data used to build the model)

In [8]:
confusion_matrix_and_metrics(y_train, y_pred_train)

{'tp': 94,
 'tn': 2601,
 'fp': 36,
 'fn': 269,
 'misclassification_rate': 0.10166666666666667,
 'false_positive_rate': 0.013651877133105802,
 'false_negative_rate': 0.7410468319559229}

### Check results on the test set (new data not yet seen by the model)

In [9]:
confusion_matrix_and_metrics(y_test, y_pred_test)

{'tp': 28,
 'tn': 862,
 'fp': 12,
 'fn': 98,
 'misclassification_rate': 0.11,
 'false_positive_rate': 0.013729977116704805,
 'false_negative_rate': 0.7777777777777778}

# Discussion Questions

### Based on the misclassification rate of your model, discuss your confidence in the ability to predict a bot. 

The tuned model reaches an 11.0% misclassification rate on the test set, close to its 10.2% training error, so it isn't badly overfit and should generalize similarly to new data. However, the base rate of bots in this data is only about 12.5%, so a model that almost always predicts "human" would already achieve roughly 12-13% misclassification without learning anything useful. Because our test false negative rate is still high (77.8%), the model is mostly succeeding by being conservative rather than by reliably recognizing bots. Overall accuracy alone overstates how confident we should be in this model's ability to actually catch bot accounts; a recall-focused metric would give a more honest picture.

### What are potential ramifications of false positives from the model?

A false positive means a real human account gets labeled as a bot. This could lead to the account being suspended, rate-limited, or shadow-banned, cutting off a legitimate user (or business) from the platform. It creates real harm and frustration, damages trust in the platform's moderation, generates support/appeal burden, and in more disproportionate cases could raise fairness or PR concerns if certain user groups are flagged more often than others.

### What are potential ramifications of false negatives from the model?

A false negative means an actual bot account goes undetected and keeps operating normally. Given our model's false negative rate of about 78%, this is currently the bigger practical risk: most bots slip through. Undetected bots can spread spam or misinformation, artificially inflate follower/engagement counts, manipulate trending topics, run scams or fake reviews, and undermine advertisers' and users' trust in the platform's metrics. Because they're a small share of accounts but potentially high-impact, letting most of them through limits how useful this model is for actually protecting the platform, even though its overall misclassification rate looks low.